In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

: 

<h2>Metrics for each video</h2>

In [ ]:
df_agg = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\Ken_lee youtube data\Aggregated_Metrics_By_Video.csv").iloc[1:,:]
df_agg.head()

In [ ]:
df_agg.info()

In [ ]:
df_agg.columns = ['Video','Video_title','Video_publish_time','Comments_added','Shares','Dislikes','Likes',
                      'Subscribers_lost','Subscribers_gained','RPM(USD)','CPM(USD)','Average_%_viewed','Average_view_duration',
                      'Views','Watch_time (hours)','Subscribers','Your_estimated_revenue(USD)','Impressions','Impressions_ctr(%)']
    

In [ ]:
df_agg['Video_publish_time'] = pd.to_datetime(df_agg['Video_publish_time'], format='mixed')#converting to datetime
df_agg.info()



In [ ]:
print(df_agg['Video_publish_time'])

In [ ]:
print(df_agg['Average_view_duration'])

In [ ]:

#df_agg['Average_view_duration'] = df_agg['Average_view_duration'].apply(lambda x: datetime.strptime(str(x), '%H:%M:%S'))
#df_agg['Avg_duration_sec'] = df_agg['Average_view_duration'].apply(lambda x: x.second + x.minute*60 + x.hour*3600)


<h4>viewer Engagement + subs gained </h4>

In [ ]:
df_agg['Engagement_ratio'] = (df_agg['Comments_added'] + df_agg['Shares'] + df_agg['Dislikes'] + df_agg['Likes']) /df_agg.Views

In [ ]:
df_agg['Views/sub_gained'] =   df_agg['Views'] / df_agg['Subscribers_gained'] 

In [ ]:
df_agg.sort_values('Video_publish_time', ascending=False, inplace=True)

<h4>Loading other datasets</h4>

In [ ]:
df_agg_sub = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\Ken_lee youtube data\Aggregated_Metrics_By_Country_And_Subscriber_Status.csv")

In [ ]:
df_comments = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\Ken_lee youtube data\All_Comments_Final.csv")

In [ ]:
df_time = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\Ken_lee youtube data\Video_Performance_Over_Time.csv")
df_time['Date'] = pd.to_datetime(df_time['Date'], format='mixed')

<h4>additional data enigneering for aggregated data</h4>

In [ ]:
df_agg_diff = df_agg.copy()
df_agg_diff.head()

videos published within the last 12 months

In [ ]:
metric_date_12mo = df_agg_diff['Video_publish_time'].max() - pd.DateOffset(months=12)

# Filter rows
filtered_df = df_agg_diff[df_agg_diff['Video_publish_time'] >= metric_date_12mo]

# Select only numeric columns
numeric_cols = filtered_df.select_dtypes(include=[np.number])

# Compute median across rows (axis=0 → column-wise median)
median_agg = np.median(numeric_cols, axis=0)

print(median_agg)


Numeric Columns

In [ ]:
numeric_cols = np.array((df_agg_diff.dtypes == 'float64') | df_agg_diff.dtypes == 'int64') #intergers and floats
df_agg_diff.iloc[:, numeric_cols] = (df_agg_diff.iloc[:, numeric_cols])

Merging df_time + df_agg dataframe

In [ ]:
# Check column names
print(df_time.columns)
print(df_agg.columns)

In [ ]:
# Ensure datetime format
df_time['Date'] = pd.to_datetime(df_time['Date'])
df_agg['Video_publish_time'] = pd.to_datetime(df_agg['Video_publish_time'])

In [ ]:
# Merge safely
df_time_diff = pd.merge(df_time, df_agg,left_on = 'External Video ID', right_on='Video', how='left')
df_time_diff['days_published'] = (df_time_diff['Date'] - df_time_diff['Video_publish_time']).dt.days

In [ ]:
df_time_diff.head()

12 months of Data

In [ ]:
date_12mo = df_agg['Video_publish_time'].max() - pd.DateOffset(months=12)
df_time_diff_yr =  df_time_diff[df_time_diff['Video_publish_time'] >= date_12mo]

Create pivot table grouped by days published

In [ ]:
print(df_time_diff_yr.columns)

In [ ]:
views_days = pd.pivot_table(
    df_time_diff_yr,
    index='days_published',
    values='Views_x',
    aggfunc=[np.mean, np.median, lambda x: np.percentile(x, 80), lambda x: np.percentile(x, 20)]
).reset_index()


In [ ]:
views_days

filter to first 30 days

In [ ]:
views_days = views_days[views_days['days_published'].between(0, 30)]

#Create cumulative views DataFrame
views_cumulative = views_days.copy()
views_cumulative[['median_views', 'pct_80', 'pct_20']] = views_cumulative[['median_views', 'pct_80', 'pct_20']].cumsum()
